# Compressão de tokens

Notebook que percorre **etapa a etapa** o pipeline de [`compressores.py`](../src/compressores.py):


Contrato:
**$F \in \mathbb{R}^{T \times D}$ → orçamento $\rho$ → seleção de $T'$ linhas → $(S,\, F[S],\, \texttt{info})$**

Cada operador define **como** escolher quais tokens manter; $\rho$ define **quantos** (percentualmente). 

Nos métodos SVD, $k$ (subespaço truncado a 95% de energia) é independente de $\rho$.

Pré-requisito: matriz $F$ construída em [`extracao_embedding.ipynb`](extracao_embedding.ipynb). Métricas ($C$, $E_k$, $R_k$, acurácia) em [`avaliacao_metricas.ipynb`](avaliacao_metricas.ipynb). Grade completa em [`trabalho_final.ipynb`](trabalho_final.ipynb).


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

_raiz = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "notebook_setup.py").exists()
)
sys.path.insert(0, str(_raiz))
import notebook_setup
notebook_setup.configurar()

import dados
import embeddings
import compressores as cp
from compressores import COMPRESSORES
from constantes import MAX_AMOSTRAS, MAX_TOKENS, N_POR_CLASSE, ORCAMENTOS, SEED

RHO_DEMO = 0.5

print("Diretório de trabalho:", os.getcwd())

## 1. Entrada $F$ — uma review de exemplo

Carregamos o corpus IMDb e extraímos $F$ da primeira review com `texto_para_embedding` (mesmo atalho de [`experimento.py`](../src/experimento.py)). Cada **linha** de $F$ é um token de conteúdo; $D=384$.


In [ ]:
textos, rotulos = dados.carregar_imdb(
    n_por_classe=N_POR_CLASSE, max_amostras=MAX_AMOSTRAS, semente_split=SEED,
)
texto_exemplo = textos[0]
F = embeddings.texto_para_embedding(texto_exemplo, max_tokens=MAX_TOKENS)
T, D = F.shape

print(f"Corpus: {len(textos)} reviews | exemplo: rótulo={rotulos[0]}")
print(f"F exemplo, shape (T x D): ({T}, {D})")

> Neste caso nÃo houve padding, mas dado o MAX_TOKENS = 256, conseguimos ver apenas a remoÇÃo dentro de texto_para_embedding do [CLS] e [SEP]


## 2. Orçamento $\rho$ — `_tokens_a_manter`

Dado orçamento $\rho \in (0, 1]$, o número de linhas mantidas é:

$$T' = \max\left(1,\ \lfloor \rho T \rfloor\right)$$

A função interna `_tokens_a_manter` centraliza essa conversão; usamos `ORCAMENTOS` do experimento.


In [ ]:
print(f"T = {T} tokens na review de exemplo\n")
for rho in ORCAMENTOS:
    t_manter = cp._tokens_a_manter(T, rho)
    print(f"  rho={rho:5.3f}  ->  T'={t_manter:3d}  ({t_manter/T:.1%} mantidos)")

n_manter = cp._tokens_a_manter(T, RHO_DEMO)
print(f"\nUsaremos rho={RHO_DEMO} -> T'={n_manter} no restante do notebook.")

## 3. Seleção por pontuação — `selecionar_sob_orcamento`

Vários métodos atribuem uma pontuação $s_i$ a cada linha e mantêm os $T'$ índices com **maior** $s_i$. A função usa `argsort` descendente e devolve os índices **ordenados** (convenção do módulo).


In [ ]:
pontuacoes_demo = np.array([0.1, 0.9, 0.3, 0.7, 0.5, 0.2])
indices_demo = cp.selecionar_sob_orcamento(pontuacoes_demo, n_manter=3)

print("Pontuações:", pontuacoes_demo)
print("Top-3 índices (ordenados):", indices_demo)
print("Valores selecionados:", pontuacoes_demo[indices_demo])

## 4. `full` — `comprimir_full`

Baseline de referência: **não poda** nenhuma linha, independentemente de $\rho$. Retorna cópia de $F$ e todos os índices $0, \ldots, T-1$.


In [ ]:
indices_full, F_full, info_full = cp.comprimir_full(F, orcamento=RHO_DEMO)

print(f"T'={len(indices_full)} (esperado T={T})")
print(f"F_full shape: {F_full.shape}")
print(f"info: {info_full}")

## 5. `random` — `comprimir_random`

Amostra **$T'$ índices uniformes** sem reposição. A `semente` fixa o sorteio (reprodutibilidade na grade experimental).


In [ ]:
indices_r1, F_r1, _ = cp.comprimir_random(F, orcamento=RHO_DEMO, semente=42)
indices_r2, _, _ = cp.comprimir_random(F, orcamento=RHO_DEMO, semente=42)
indices_r3, _, _ = cp.comprimir_random(F, orcamento=RHO_DEMO, semente=99)

print(f"T'={len(indices_r1)} | primeiros índices: {indices_r1[:10]}...")
print(f"Mesma semente -> iguais: {np.array_equal(indices_r1, indices_r2)}")
print(f"Semente diferente -> distintos: {not np.array_equal(indices_r1, indices_r3)}")

## 6. SVD truncada — `_svd_decomposicao`

Decomposição $F = U \Sigma V^\top$. Escolhemos $k$ como o menor posto que captura **95% da energia espectral** $\sum_j \sigma_j^2$. Valores singulares abaixo de `TOL_SINGULAR` são zerados.


In [ ]:
U, S, Vt, k, energia_explicada = cp._svd_decomposicao(F, variancia_explicada=0.95)
energia = S ** 2
energia_acum = np.cumsum(energia) / np.sum(energia)

print(f"k = {k}  |  energia_explicada = {energia_explicada:.4f}")
print(f"sigma_1 = {S[0]:.3f}  |  sigma_k = {S[k-1]:.3f}  |  rank efetivo = {np.sum(S > 0)}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(1, len(S) + 1), energia_acum, marker=".", markersize=4)
ax.axvline(k, color="tab:red", linestyle="--", label=f"k={k} (95% energia)")
ax.axhline(0.95, color="gray", linestyle=":", alpha=0.7)
ax.set_xlabel("j (índice singular)")
ax.set_ylabel("energia acumulada")
ax.set_title("Escolha de k na SVD truncada")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Leverage clássico — `pontuar_leverage`

**Statistical leverage** no subespaço das $k$ primeiras colunas de $U$:

$$\ell_i = \frac{1}{k} \sum_{j=1}^{k} U_{ij}^2$$

Tokens com maior $\ell_i$ são mais "influentes" na geometria global de $F$.


In [ ]:
leverage = cp.pontuar_leverage(U, k)
top_idx = np.argsort(leverage)[::-1][:5]

print("Top-5 tokens por leverage:")
for rank, i in enumerate(top_idx, 1):
    print(f"  {rank}. índice={i:3d}  ell_i={leverage[i]:.4f}")

In [ ]:
texto_exemplo

In [ ]:
# Mapear linha i de F -> subtoken legível
tokenizer, _ = embeddings.carregar_modelo()
entradas = embeddings.tokenizar_texto(texto_exemplo, tokenizer, max_tokens=MAX_TOKENS)
tokens = embeddings.obter_tokens(tokenizer, entradas)
pos_conteudo = embeddings.indices_tokens_conteudo(tokenizer, entradas)
tokens_conteudo = [tokens[p] for p in pos_conteudo]

print("Top-5 tokens por leverage:")
for rank, i in enumerate(top_idx, 1):
    tok = tokens_conteudo[i]
    # posição na sequência original do modelo (com [CLS]/[SEP])
    pos_original = pos_conteudo[i]
    print(f"  {rank}. índice_F={i:3d}  pos_modelo={pos_original:3d}  "
          f"ell_i={leverage[i]:.4f}  token={tok!r}")

> Conseguimos ver quem sÃo esses tokens definidos como maior leverage score


## 8. Leverage + energia — `pontuar_leverage_energia`

Mesma ideia, mas cada direção singular é ponderada por $\sigma_j^2$ (energia da componente):

$$\ell_i^{\mathrm{energia}} = \sum_{j=1}^{k} U_{ij}^2 \cdot \sigma_j^2$$

Comparação visual com o leverage clássico nos tokens de maior pontuação.


In [ ]:
leverage_en = cp.pontuar_leverage_energia(U, S, k)

top_lev = set(np.argsort(leverage)[::-1][:n_manter])
top_en = set(np.argsort(leverage_en)[::-1][:n_manter])
print(f"Top-{n_manter} índices em comum: {len(top_lev & top_en)} de {n_manter}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, scores, titulo in [
    (axes[0], leverage, "pontuar_leverage"),
    (axes[1], leverage_en, "pontuar_leverage_energia"),
]:
    top = np.argsort(scores)[::-1][:n_manter]
    ax.bar(range(n_manter), scores[top], color="tab:blue")
    ax.set_xticks(range(n_manter))
    ax.set_xticklabels([str(i) for i in top], rotation=45, fontsize=8)
    ax.set_xlabel("índice do token (top T')")
    ax.set_title(titulo)
axes[0].set_ylabel("pontuação")
plt.suptitle(f"Comparação de leverage (rho={RHO_DEMO}, k={k})")
plt.tight_layout()
plt.show()

In [ ]:
top_en_idx = np.argsort(leverage_en)[::-1][:5]

print("Top-5 tokens por leverage + energia:")
for rank, i in enumerate(top_en_idx, 1):
    tok = tokens_conteudo[i]
    pos_original = pos_conteudo[i]
    print(
        f"  {rank}. índice_F={i:3d}  pos_modelo={pos_original:3d}  "
        f"ell_i={leverage_en[i]:.4f}  token={tok!r}"
    )

## 9. `svd` — `comprimir_svd`

Pipeline completo: SVD → `pontuar_leverage` → `selecionar_sob_orcamento`. Também monta a reconstrução $F_k = U_{:,1:k}\,\mathrm{diag}(\Sigma_k)\,V_{1:k,:}^\top$ em `info['reconstrucao']` (útil para inspeção algébrica; a poda usa linhas originais $F[S]$).


In [ ]:
indices_svd, F_svd, info_svd = cp.comprimir_svd(F, orcamento=RHO_DEMO)
F_k = info_svd["reconstrucao"]

print(f"T'={len(indices_svd)} | k={info_svd['k']} | energia_explicada={info_svd['energia_explicada']:.4f}")
print(f"F_svd shape: {F_svd.shape}  |  F_k shape: {F_k.shape}")
print(f"Primeiros índices mantidos: {indices_svd[:10]}...")

In [ ]:
tokens_mantidos = [tokens_conteudo[i] for i in indices_svd]
texto_por_svd = tokenizer.convert_tokens_to_string(tokens_mantidos)

print(f"Original ({len(tokens_conteudo)} tokens):")
print(tokenizer.convert_tokens_to_string(tokens_conteudo)[:500], "...\n")

print(f"Comprimido svd ({len(indices_svd)} tokens, rho={RHO_DEMO}):")
print(texto_por_svd[:500], "...")

## 10. `svd_energia` — `comprimir_svd_energia`

Mesmo $k$ e mesma SVD; muda apenas a função de pontuação para `pontuar_leverage_energia`. Os índices selecionados podem diferir de `svd`.


In [ ]:
indices_en, F_en, info_en = cp.comprimir_svd_energia(F, orcamento=RHO_DEMO)

print(f"T'={len(indices_en)} | k={info_en['k']} (igual a svd: {info_en['k'] == info_svd['k']})")
print(f"Índices iguais a svd: {np.array_equal(indices_svd, indices_en)}")
print(f"Interseção: {len(set(indices_svd) & set(indices_en))} de {n_manter}")

In [ ]:
tokens_mantidos = [tokens_conteudo[i] for i in indices_en]
texto_por_svd = tokenizer.convert_tokens_to_string(tokens_mantidos)

print(f"Original ({len(tokens_conteudo)} tokens):")
print(tokenizer.convert_tokens_to_string(tokens_conteudo)[:500], "...\n")

print(f"Comprimido svd ({len(indices_svd)} tokens, rho={RHO_DEMO}):")
print(texto_por_svd[:500], "...")

## 11. Atalho — `COMPRESSORES`

Dicionário usado em [`experimento.py`](../src/experimento.py). Loop com $\rho$ fixo: apenas **construção** ($T$, $T'$, $k$, shapes).


In [ ]:
print(f"{'método':12s}  T'   k    shape(F_podado)")
print("-" * 42)
for nome, fn in COMPRESSORES.items():
    indices, F_pod, info = fn(F, RHO_DEMO, semente=0)
    k_str = str(info["k"]) if info["k"] is not None else "—"
    print(f"{nome:12s}  {len(indices):3d}  {k_str:>3s}  {F_pod.shape}")

## 12. $\rho$ vs $k$ — independência

Ao variar $\rho$, $T'$ muda, mas $k$ e `energia_explicada` (definidos só por $F$) permanecem constantes — propriedade central do protocolo SVD.


In [ ]:
print(f"{'rho':>6s}  T'   k    energia_explicada")
print("-" * 36)
for rho in ORCAMENTOS:
    indices, _, info = cp.comprimir_svd(F, orcamento=rho)
    print(f"{rho:6.3f}  {len(indices):3d}  {info['k']:3d}  {info['energia_explicada']:.4f}")

> Energia explicada relacionado ao k primeiros modos (colunas do SVD economico), nÃo depende de T' (nossas linhas pÓs corte baseado em rho)


## 13. Tokens mantidos — inspeção qualitativa

Mapeamos índices de $F$ de volta aos subtokens (mesmo pipeline de [`extracao_embedding.ipynb`](extracao_embedding.ipynb)) e comparamos o que `random` e `svd` preservam.


In [ ]:
tokenizer, modelo = embeddings.carregar_modelo()
entradas = embeddings.tokenizar_texto(texto_exemplo, tokenizer, max_tokens=MAX_TOKENS)
tokens = embeddings.obter_tokens(tokenizer, entradas)
pos_conteudo = embeddings.indices_tokens_conteudo(tokenizer, entradas)
tokens_conteudo = [tokens[i] for i in pos_conteudo]

def tokens_mantidos(indices_compressor, n_mostrar=12):
    return [tokens_conteudo[i] for i in indices_compressor[:n_mostrar]]

indices_random, _, _ = cp.comprimir_random(F, orcamento=RHO_DEMO, semente=0)
print(f"Total tokens de conteúdo: {len(tokens_conteudo)}\n")
print(f"random (primeiros {n_manter}): {tokens_mantidos(indices_random)}")
print(f"svd   (primeiros {n_manter}): {tokens_mantidos(indices_svd)}")
